# Modelagem baseada em embeddings

Este notebook tem como objetivo principal modelar a ocorrência de erros do tipo "Don't Go" nas próximas 6 horas.

Nessa versão usaremos LightGBM que é um modelo baseado em árvores de decisão com GradientBoosting e os dados serão baseados nas médias móveis com decaimento exponencial dos embeddings dos alarmes que aconteceram para cada máquina.

---

# Import dos dados e bibliotecas

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import pyarrow
np.random.seed(21)

In [2]:
df_caminhao = pd.read_parquet('../datasets/datasets_tratados/df_caminhao_modelagem_embeddings.parquet', engine='pyarrow')

# Tratamento dos dados para modelagem

In [3]:
# Tratamento das variáveis cíclicas para que o modelo tenha ideia de continuidade
# Por exmeplo, dia 31 de um mês vem logo antes do dia 1 de outro

# Definindo os limites para cada variável
MAX_MES = 12.0
MAX_DIA = 31.0
MAX_TEMPO_TURNO = 12 * 60 * 60 

# Transformando o mês
df_caminhao['Mes_sin'] = np.sin(2 * np.pi * df_caminhao['Mes'] / MAX_MES)
df_caminhao['Mes_cos'] = np.cos(2 * np.pi * df_caminhao['Mes'] / MAX_MES)

# Transformando o dia
df_caminhao['Dia_sin'] = np.sin(2 * np.pi * df_caminhao['Dia'] / MAX_DIA)
df_caminhao['Dia_cos'] = np.cos(2 * np.pi * df_caminhao['Dia'] / MAX_DIA)

# Transformando o tempo dentro de cada turno
df_caminhao['Tempo_Turno_sin'] = np.sin(2 * np.pi * df_caminhao['Tempo_Turno'] / MAX_TEMPO_TURNO)
df_caminhao['Tempo_Turno_cos'] = np.cos(2 * np.pi * df_caminhao['Tempo_Turno'] / MAX_TEMPO_TURNO)

# Removendo as variáveis originiais
df_caminhao.drop(columns=['Mes', 'Dia', 'Tempo_Turno'], inplace=True)

In [4]:
# Seleção de variáveis que serão usadas no modelo

# Variáveis categóricas
cat_cols = [
    'Situacao_Operacional',
    # 'Nome_Operador_Anon',
    'Criticidade',
    'Turno',
]

# Variáveis numéricas
num_cols = [
    'Mes_sin', 'Mes_cos', 
    'Dia_sin', 'Dia_cos', 
    'Tempo_Turno_sin', 'Tempo_Turno_cos', 
    'Tempo_Situacao'
]

# Transforma as variáveis categóricas no tipo 'category'
for col in cat_cols:
    df_caminhao[col] = df_caminhao[col].astype('category')

# Lista as colunas que formas os embeddings
colunas_ewma = [col for col in df_caminhao.columns if col.startswith('EWMA_')]

# Final
features = colunas_ewma + cat_cols + num_cols
target = 'Classe_Risco'

# Separação dos dados

Para garantir a generalização dos modelos, os dados foram separados em 3 conjuntos com base na divisão temporal e Holdout

O conjunto de teste é composto por toda a telemetria de 15% dos veículos, sorteados aleatoriamente antes da separação dos dados.

Já o conjunto de treino e validação são formados pelos 85% restantes dos dados. O conjunto de validação representa os 20% finais dos dados divididos temporalmente. Enquanto o conjunto de treino representa os 80% iniciais.

Dessa forma, o modelo é testado na generalização para mesmos veículos ao longo do tempo, além da generalização para cada um dos veículos contido no conjunto de teste


In [5]:
# Retorna as tags e sorteia os dados de holdout
tags_unicas = df_caminhao['TAG'].unique()
qtd_holdout = int(len(tags_unicas) * 0.15) # 15% dos caminhões para o teste cego

tags_holdout = np.random.choice(tags_unicas, size=qtd_holdout, replace=False)

# Separa a base de Holdout Absoluto e a Base de Desenvolvimento (Treino + Validação)
df_test = df_caminhao[df_caminhao['TAG'].isin(tags_holdout)].copy()
df_caminhao = df_caminhao[~df_caminhao['TAG'].isin(tags_holdout)].copy()

print(f"Caminhões no Holdout (Nunca vistos): {len(tags_holdout)}")
print(f"Caminhões no Desenvolvimento: {len(tags_unicas) - len(tags_holdout)}")

Caminhões no Holdout (Nunca vistos): 4
Caminhões no Desenvolvimento: 24


In [6]:
# Garante que está ordenado no tempo para que a sepação não seja enviesada
df_caminhao = df_caminhao.sort_values(by=['TAG', 'Data_Evento'])

# Calcula o percentil temporal de cada log dentro da história do seu respectivo caminhão
df_caminhao['Ordem_Tempo'] = df_caminhao.groupby('TAG')['Data_Evento'].rank(method='first')
df_caminhao['Total_Logs_TAG'] = df_caminhao.groupby('TAG')['TAG'].transform('count')
df_caminhao['Percentil_Tempo'] = df_caminhao['Ordem_Tempo'] / df_caminhao['Total_Logs_TAG']

# Corta em 80% (Passado) e 20% (Futuro) para cada caminhão
df_train = df_caminhao[df_caminhao['Percentil_Tempo'] <= 0.80].copy()
df_val = df_caminhao[df_caminhao['Percentil_Tempo'] > 0.80].copy()

# Limpa as colunas auxiliares
df_train.drop(columns=['Ordem_Tempo', 'Total_Logs_TAG', 'Percentil_Tempo'], inplace=True)
df_val.drop(columns=['Ordem_Tempo', 'Total_Logs_TAG', 'Percentil_Tempo'], inplace=True)

print(f"\nLinhas de Treino (Passado): {len(df_train):,}")
print(f"Linhas de Validação (Futuro): {len(df_val):,}")
print(f"Linhas de Holdout (Teste Cego): {len(df_test):,}")


Linhas de Treino (Passado): 781,853
Linhas de Validação (Futuro): 195,476
Linhas de Holdout (Teste Cego): 226,527


In [7]:
# Treino e Validação
X_train, y_train = df_train[features], df_train[target]
X_val, y_val = df_val[features], df_val[target]

# Holdout
X_test, y_test = df_test[features], df_test[target]

# Modelagem

In [8]:
# Usa o LabelEnconder para transformar a coluna target para números
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

y_train

array([1, 1, 1, ..., 0, 0, 0], shape=(781853,))

In [9]:
# Como é preferível que o modelo seja pessimista, vamos dar mais peso para a classe que indica que o caminhão está prestes a quebrar

pesos = {
    0: 3.4,
    1: 1.0
}

In [10]:
# Define os parâmetros iniciais do modelo
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    objective='binary',
    metric='binary_logloss',
    class_weight=pesos,
    learning_rate=0.01,
    random_state=42,
    colsample_bytree=0.3,
    max_depth=3,
    num_leaves=15,
    verbose=-1
)

# Adiciona o EarlyStopping com base na validação
model = lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=['train', 'validation'],
    eval_metric='binary_logloss',
    callbacks=[lgb.early_stopping(stopping_rounds=15)] # Para o treino se a validação parar de melhorar
)

Training until validation scores don't improve for 15 rounds
Did not meet early stopping. Best iteration is:
[100]	train's binary_logloss: 0.522327	validation's binary_logloss: 0.631203


In [11]:
# Printa o report de classificação do modelo para que possamos avaliar as métricas para cada uma das classes

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.60      0.82      0.69     65207
           1       0.91      0.78      0.84    161320

    accuracy                           0.79    226527
   macro avg       0.76      0.80      0.77    226527
weighted avg       0.82      0.79      0.80    226527



In [12]:
# Recupera a importância das variáveis para as predições do modelo

importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance_df.head(25)

,Feature,Importance
15,EWMA_2h_Dim_15,67
16,EWMA_6h_Dim_0,44
0,EWMA_2h_Dim_0,33
8,EWMA_2h_Dim_8,29
24,EWMA_6h_Dim_8,27
25,EWMA_6h_Dim_9,24
31,EWMA_6h_Dim_15,23
9,EWMA_2h_Dim_9,22
32,EWMA_24h_Dim_0,22
6,EWMA_2h_Dim_6,22


In [13]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[ 53355  11852]
 [ 35264 126056]]
